In [10]:
# ── Cell 1: Imports & paths ───────────────────────────────────────────────
import polars as pl
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from statsmodels.nonparametric.smoothers_lowess import lowess
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
Y_MEDIAN = 10_784  # locked from EDA
YIELD_FLOOR   = 6_000
YIELD_CEILING = 15_000

In [11]:
# ── Cell 2: Load & inspect enriched quotes ────────────────────────────────
df = pl.read_parquet(PROCESSED_DIR / "quotes_deduped_enriched.parquet")
print(df.schema)
print(df.shape)
df.head(5)

Schema({'QUOTE_LEG_ID': Int64, 'quote_date': Date, 'dep_date': Date, 'eft_hrs': Float64, 'revenue': Float64, 'booked': Int8, 'final_status': String, 'dep_icao': String, 'arr_icao': String, 'fleet': String, 'status_history_count': UInt32, 'callout_days': Int64, 'yield_per_eft': Float64, 'leg_yield': Float64, 'final_yield': Float64})
(47400, 15)


QUOTE_LEG_ID,quote_date,dep_date,eft_hrs,revenue,booked,final_status,dep_icao,arr_icao,fleet,status_history_count,callout_days,yield_per_eft,leg_yield,final_yield
i64,date,date,f64,f64,i8,str,str,str,str,u32,i64,f64,f64,f64
8806242,2024-01-01,2024-01-03,5.083333,30350.0,0,"""Cancelled""","""KTVL""","""KTEB""","""Challenger 300""",4,2,5970.492195,null,5970.492195
8806090,2024-01-01,2024-01-09,2.25,null,1,"""Unconfirmed""","""MDPC""","""KOPF""","""Challenger 300""",14,8,null,11120.498276,11120.498276
8805715,2024-01-01,2024-01-05,2.75,39200.0,0,"""Cancelled""","""KJAC""","""KCLL""","""Challenger 300""",3,4,14254.545455,null,14254.545455
8808838,2024-01-01,2024-01-04,3.916667,39850.0,0,"""Cancelled""","""KASE""","""KMIA""","""Challenger 300""",4,3,10174.467219,null,10174.467219
8804396,2024-01-01,2024-01-17,4.083333,null,1,"""Confirmed""","""KBOS""","""KEGE""","""Challenger 300""",13,16,null,null,null


In [12]:
# ── Cell 3 PATCHED: correct yield column for each elasticity ─────────────

YIELD_COL_QC   = "yield_per_eft"   # quoted offer price — correct for quote count
YIELD_COL_CONV = "final_yield"     # covers both booked & unbooked — correct for conversion

# ── Quote Count Elasticity bins (unchanged — yield_per_eft is right here) ─
df_qc = (
    df
    .filter(
        (pl.col(YIELD_COL_QC) >= 7_000) &
        (pl.col(YIELD_COL_QC) <= 14_750) &
        pl.col(YIELD_COL_QC).is_not_null()
    )
    .with_columns([
        (pl.col(YIELD_COL_QC) / 500).floor().cast(pl.Int32).alias("bin_id")
    ])
    .with_columns([
        (pl.col("bin_id") * 500 + 250).alias("yield_mid")
    ])
)

df_qc_binned = (
    df_qc
    .group_by(["quote_date", "yield_mid"])
    .agg(pl.len().alias("count"))
    .group_by("yield_mid")
    .agg(pl.col("count").mean().alias("avg_count"))
    .sort("yield_mid")
)

print("Quote count bins:")
print(df_qc_binned.sort("yield_mid").to_pandas().to_string())

# ── Conversion Elasticity bins (FIXED — use final_yield) ─────────────────
df_conv = (
    df
    .filter(
        (pl.col(YIELD_COL_CONV) >= 5_000) &
        (pl.col(YIELD_COL_CONV) <= 20_000) &
        pl.col(YIELD_COL_CONV).is_not_null()
    )
    .with_columns([
        (pl.col(YIELD_COL_CONV) / 1_000).floor().cast(pl.Int32).alias("bin_id")
    ])
    .with_columns([
        (pl.col("bin_id") * 1_000 + 500).alias("yield_mid")
    ])
)

df_conv_binned = (
    df_conv
    .group_by("yield_mid")
    .agg([
        pl.len().alias("total_quotes"),
        pl.col("booked").sum().alias("booked_quotes")
    ])
    .with_columns([
        (pl.col("booked_quotes") / pl.col("total_quotes")).alias("conversion_rate")
    ])
    .filter(pl.col("total_quotes") >= 10)
    .sort("yield_mid")
)

print("\nConversion bins (final_yield):")
print(df_conv_binned.sort("yield_mid").to_pandas().to_string())

Quote count bins:
    yield_mid  avg_count
0        7250   3.535777
1        7750   4.070033
2        8250   4.398400
3        8750   3.872611
4        9250   4.259141
5        9750   4.335385
6       10250   4.363919
7       10750   4.270701
8       11250   3.775748
9       11750   3.517529
10      12250   3.165232
11      12750   3.084270
12      13250   2.804734
13      13750   2.608000
14      14250   2.308390
15      14750   1.795539

Conversion bins (final_yield):
    yield_mid  total_quotes  booked_quotes  conversion_rate
0        5500           488            153         0.313525
1        6500          1880            257         0.136702
2        7500          4830            305         0.063147
3        8500          5481            300         0.054735
4        9500          5748            251         0.043667
5       10500          5700            212         0.037193
6       11500          4509            129         0.028609
7       12500          3572             86   

In [13]:
# ── Cell 4: Fit ε_q(Y) — LOWESS ──────────────────────────────────────────
qc_pd = df_qc_binned.to_pandas().sort_values("yield_mid")
x_qc = qc_pd["yield_mid"].values.astype(float)
y_qc = qc_pd["avg_count"].values.astype(float)

# LOWESS fit — frac=0.5 (smooth, appropriate for flat signal)
lowess_fit = lowess(y_qc, x_qc, frac=0.5, return_sorted=True)
x_low = lowess_fit[:, 0]
y_low = lowess_fit[:, 1]

# Normalise: ε_q(Y) = smoothed_count(Y) / smoothed_count(Y_median)
# Use np.interp to evaluate at Y_median
y_low_at_median = np.interp(Y_MEDIAN, x_low, y_low)
y_low_norm = y_low / y_low_at_median

print(f"LOWESS value at Y_median ({Y_MEDIAN}): {y_low_at_median:.2f}")
print(f"Normalised range: [{y_low_norm.min():.3f}, {y_low_norm.max():.3f}]")
print(f"ε_q at Y_median (should be 1.0): {np.interp(Y_MEDIAN, x_low, y_low_norm):.4f}")

LOWESS value at Y_median (10784): 4.01
Normalised range: [0.482, 1.056]
ε_q at Y_median (should be 1.0): 1.0000


In [14]:
# ── Cell 5 FINAL: prep + fit ε_c(Y) ─────────────────────────────────────

# Step 1: extract numpy arrays from df_conv_binned (THIS WAS MISSING)
conv_pd  = df_conv_binned.to_pandas().sort_values("yield_mid")
x_cv     = conv_pd["yield_mid"].values.astype(float)
y_cv     = conv_pd["conversion_rate"].values.astype(float)
x_cv_k   = x_cv / 1_000          # rescale to $k — [5.5, 6.5, ..., 19.5]
Y_MEDIAN_K = Y_MEDIAN / 1_000    # 10.784

# Step 2: exclude the cliff bins below $7.5k (different pricing regime)
mask        = x_cv_k >= 7.5
x_cv_k_fit  = x_cv_k[mask]
y_cv_fit    = y_cv[mask]

print("Bins used for fitting:")
for xv, yv in zip(x_cv_k_fit, y_cv_fit):
    print(f"  ${xv*1000:,.0f}  →  {yv:.4f}")

# Step 3: fit a * exp(-b * Y_k)
def exp_decay(Y_k, a, b):
    return a * np.exp(-b * Y_k)

# Informed guesses: at $7.5k conv=0.063, at $19.5k conv=0.010
# exp(-b×12) = 0.010/0.063 → b ≈ 0.15, a ≈ 0.8
p0     = [0.8, 0.15]
bounds = ([0, 0], [np.inf, np.inf])

popt, pcov = curve_fit(exp_decay, x_cv_k_fit, y_cv_fit,
                       p0=p0, bounds=bounds, maxfev=10_000)
a_fit, b_fit = popt
perr = np.sqrt(np.diag(pcov))

print(f"\nFitted:  a={a_fit:.4f} (±{perr[0]:.4f}),  b={b_fit:.4f} (±{perr[1]:.4f})  [Y in $k]")

conv_at_median = exp_decay(Y_MEDIAN_K, a_fit, b_fit)
print(f"\nConversion at Y_median:          {conv_at_median:.4f}  (expect ~0.037)")
print(f"ε_c at Y_median (should be 1.0): {exp_decay(Y_MEDIAN_K,  a_fit, b_fit) / conv_at_median:.4f}")
print(f"ε_c at floor  $6,000:            {exp_decay(6.0,  a_fit, b_fit) / conv_at_median:.4f}  (expect 2–4x)")
print(f"ε_c at ceiling $18,000:          {exp_decay(18.0, a_fit, b_fit) / conv_at_median:.4f}  (expect 0.3–0.5x)")

y_pred_fit = exp_decay(x_cv_k_fit, a_fit, b_fit)
ss_res = np.sum((y_cv_fit - y_pred_fit)**2)
ss_tot = np.sum((y_cv_fit - y_cv_fit.mean())**2)
r2     = 1 - ss_res / ss_tot
print(f"\nR²: {r2:.4f}  (expect >0.85)")

Bins used for fitting:
  $7,500  →  0.0631
  $8,500  →  0.0547
  $9,500  →  0.0437
  $10,500  →  0.0372
  $11,500  →  0.0286
  $12,500  →  0.0241
  $13,500  →  0.0166
  $14,500  →  0.0185
  $15,500  →  0.0149
  $16,500  →  0.0147
  $17,500  →  0.0141
  $18,500  →  0.0148
  $19,500  →  0.0102

Fitted:  a=0.2297 (±0.0237),  b=0.1736 (±0.0102)  [Y in $k]

Conversion at Y_median:          0.0353  (expect ~0.037)
ε_c at Y_median (should be 1.0): 1.0000
ε_c at floor  $6,000:            2.2948  (expect 2–4x)
ε_c at ceiling $18,000:          0.2857  (expect 0.3–0.5x)

R²: 0.9731  (expect >0.85)


In [15]:
# ── Cell 6: Validation plots (3-panel) ───────────────────────────────────
Y_range = np.linspace(YIELD_FLOOR, YIELD_CEILING, 500)

# Elasticity functions for plotting
def eps_q(Y):
    return np.interp(Y, x_low, y_low_norm)

def eps_c(Y):
    return exp_decay(Y / 1_000, a_fit, b_fit) / conv_at_median

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "ε_q(Y) — Quote Count Elasticity (LOWESS)",
        "ε_c(Y) — Conversion Elasticity (Exp Decay)",
        "D(Y) = Quotes × ε_q(Y) × ε_c(Y) — Full Demand Curve"
    ],
    horizontal_spacing=0.10
)

# ── Panel 1: ε_q LOWESS ──────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=x_qc, y=y_qc / y_low_at_median,
    mode="markers", marker=dict(size=7, color="steelblue", opacity=0.7),
    name="Binned quote counts (norm)"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x_low, y=y_low_norm,
    mode="lines", line=dict(color="crimson", width=2.5),
    name="LOWESS fit"
), row=1, col=1)

fig.add_vline(x=Y_MEDIAN, line_dash="dash", line_color="green",
              annotation_text=f"Y_med=${Y_MEDIAN:,}", row=1, col=1)
fig.add_hline(y=1.0, line_dash="dot", line_color="gray", row=1, col=1)

# ── Panel 2: ε_c exponential ─────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=x_cv, y=y_cv / conv_at_median,
    mode="markers", marker=dict(size=7, color="steelblue", opacity=0.7),
    name="Binned conversion (norm)"
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=Y_range, y=eps_c(Y_range),
    mode="lines", line=dict(color="crimson", width=2.5),
    name=f"a·exp(−b·Y)  R²={r2:.3f}"
), row=1, col=2)

fig.add_vline(x=Y_MEDIAN, line_dash="dash", line_color="green",
              annotation_text=f"Y_med=${Y_MEDIAN:,}", row=1, col=2)
fig.add_hline(y=1.0, line_dash="dot", line_color="gray", row=1, col=2)

# ── Panel 3: D(Y) composite demand curve ─────────────────────────────────
mean_quotes = 80.3   # from 07 forecast
D_Y = mean_quotes * eps_q(Y_range) * eps_c(Y_range)

fig.add_trace(go.Scatter(
    x=Y_range, y=D_Y,
    mode="lines", line=dict(color="darkorange", width=3),
    name="D(Y) demand curve"
), row=1, col=3)

# Mean OD supply line — this is where the solver will clear
fig.add_hline(y=1.50, line_dash="dash", line_color="navy",
              annotation_text="Mean supply=1.50", row=1, col=3)

# Mark approximate clearing yield
D_at_range = mean_quotes * eps_q(Y_range) * eps_c(Y_range)
# Find where D(Y) first crosses 1.5 from above
crossing_idx = np.where(D_at_range <= 1.50)[0]
if len(crossing_idx) > 0:
    Y_approx_clear = Y_range[crossing_idx[0]]
    fig.add_vline(x=Y_approx_clear, line_dash="dot", line_color="purple",
                  annotation_text=f"~clearing ${Y_approx_clear:,.0f}",
                  row=1, col=3)

fig.add_vline(x=Y_MEDIAN, line_dash="dash", line_color="green",
              annotation_text=f"Y_med=${Y_MEDIAN:,}", row=1, col=3)

fig.update_layout(
    width=1200, height=700,
    title="08 — Elasticity Curves + Composite Demand",
    legend=dict(orientation="h", x=0.5, xanchor="center",
                y=-0.10, font=dict(size=10)),
    margin=dict(l=50, r=50, t=50, b=50)
)
fig.update_yaxes(title_text="ε_q (normalised)", row=1, col=1)
fig.update_yaxes(title_text="ε_c (normalised)", row=1, col=2)
fig.update_yaxes(title_text="D(Y) — effective quote demand", row=1, col=3)
fig.update_xaxes(title_text="Yield ($/EFT hr)", tickformat="$,.0f")

fig.show()
fig.write_html("./08_elasticity_curves.html")
print("Saved → ./08_elasticity_curves.html")

Saved → ./08_elasticity_curves.html


In [16]:
# ── Cell 7: Save elasticity_params.json ──────────────────────────────────
elasticity_params = {
    "Y_median": Y_MEDIAN,
    "yield_floor": YIELD_FLOOR,
    "yield_ceiling": YIELD_CEILING,
    "quote_count_elasticity": {
        "method": "lowess",
        "frac": 0.5,
        "x": x_low.tolist(),
        "y_normalized": y_low_norm.tolist()
    },
    "conversion_elasticity": {
        "method": "exponential",
        "formula": "a * exp(-b * Y), normalised at Y_median",
        "a": float(a_fit),
        "b": float(b_fit),
        "r2": float(r2)
    }
}

elasticity_params["conversion_elasticity"]["b_unit"] = "per_$k"
elasticity_params["conversion_elasticity"]["note"] = "Y must be divided by 1000 before applying b"

out_path = PROCESSED_DIR / "elasticity_params.json"
with open(out_path, "w") as f:
    json.dump(elasticity_params, f, indent=2)

print(f"✅ Saved → {out_path}")
print(f"\nKey params:")
print(f"  a = {a_fit:.4f}")
print(f"  b = {b_fit:.6f}")
print(f"  R² = {r2:.4f}")
print(f"  LOWESS x range: [{x_low.min():.0f}, {x_low.max():.0f}]")
print(f"  LOWESS points: {len(x_low)}")

✅ Saved → ../data/processed/elasticity_params.json

Key params:
  a = 0.2297
  b = 0.173633
  R² = 0.9731
  LOWESS x range: [7250, 14750]
  LOWESS points: 16
